In [1]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
data = pd.read_csv('MFCCoutput.csv')
data = data.drop(data.columns[0], axis = 1)
print(data.head())

           0           1          2          3          4          5  \
0 -398.97055   92.846680  -6.212745  19.836878  -3.015201   9.478265   
1 -232.39255  115.044525 -21.028315  39.190132 -17.016842   7.619745   
2 -466.48450   89.272385  -8.458461  30.776363 -11.168960  18.305796   
3 -466.73505   62.805060  12.439709  29.304922  12.614448   9.676723   
4 -426.44970   80.985410  -4.792509  36.350883  -0.092283  19.156744   

          6          7         8         9  ...       119       120       121  \
0  5.134083   6.716100  1.437088  0.016323  ...  0.018726 -0.244671 -0.650705   
1 -2.724971   4.140653 -1.230497 -0.977595  ...  0.197640  0.564010  0.166091   
2  0.989266  10.417193  0.574027  1.563966  ...  0.232031  0.074036  0.237405   
3  1.418015  13.074185  0.037665  3.573357  ...  0.408432 -0.018611 -0.274066   
4 -5.060070  10.123955  2.196594  1.254182  ... -0.064134  0.120973  0.070115   

        122       123       124       125       126       127  class  
0 -0.2622

In [4]:
importance_df = pd.read_csv('feature_importance_basic.csv')
print(importance_df.head())
feature_names = importance_df["Feature"].tolist()
print(feature_names)


   Feature  CB_Importance  LGBM_Importance  ET_Importance  Avg_Importance
0        0       1.000000         1.000000       1.000000        1.000000
1       36       0.642909         0.740157       0.766617        0.716561
2       82       0.484306         0.732283       0.782581        0.666390
3       49       0.290911         0.787402       0.672607        0.583640
4       12       0.293238         0.464567       0.949464        0.569090
[0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 60, 93, 81, 38, 7, 62, 29, 61, 56, 118, 90, 70, 107, 40, 100, 46, 87, 1, 89, 63, 66, 102, 76, 59, 99, 15, 71, 65, 34, 19, 103, 101, 41, 54, 43, 28, 33, 104, 6, 106, 53, 96, 115, 68, 98, 94, 20, 113, 124, 2, 79, 52, 119, 95, 78, 64, 86, 11, 50, 25, 111, 91, 32, 114, 31, 8, 58, 120, 92, 10, 27, 39, 4, 22, 9, 97, 17, 126, 48, 51, 105, 5, 18, 109, 13, 74, 84, 121, 75, 116, 80, 122, 73, 123, 127, 125, 110]


In [5]:
iteration = 1
for i in range(10, len(feature_names)+1 , 10 ):
    print(f'{iteration}: {feature_names[:i]}')
    iteration += 1

1: [0, 36, 82, 49, 12, 24, 14, 83, 117, 16]
2: [0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77]
3: [0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112]
4: [0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 60, 93, 81, 38, 7, 62, 29, 61, 56]
5: [0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 60, 93, 81, 38, 7, 62, 29, 61, 56, 118, 90, 70, 107, 40, 100, 46, 87, 1, 89]
6: [0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 60, 93, 81, 38, 7, 62, 29, 61, 56, 118, 90, 70, 107, 40, 100, 46, 87, 1, 89, 63, 66, 102, 76, 59, 99, 15, 71, 65, 34]
7: [0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 6

In [6]:
performance_list = {}

for i in range(10, len(feature_names)+1 , 10 ):
    selected_features = feature_names[:i]
    print(selected_features)

    drop_columns = []

    for j in range(0,128):
        if j not in selected_features:
            drop_columns.append(j)

    drop_columns = list(map(str, drop_columns))    

    df_top_k = data.drop(columns=drop_columns, axis = 1)

    X = df_top_k.drop("class", axis=1)
    y = df_top_k["class"]


    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = CatBoostClassifier()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    performance_list.update({i:accuracy})




[0, 36, 82, 49, 12, 24, 14, 83, 117, 16]
Learning rate set to 0.017484
0:	learn: 0.6694507	total: 71.7ms	remaining: 1m 11s
1:	learn: 0.6439846	total: 77.1ms	remaining: 38.5s
2:	learn: 0.6221629	total: 81.2ms	remaining: 27s
3:	learn: 0.6016179	total: 86.9ms	remaining: 21.6s
4:	learn: 0.5827031	total: 91.6ms	remaining: 18.2s
5:	learn: 0.5628242	total: 96.4ms	remaining: 16s
6:	learn: 0.5457374	total: 103ms	remaining: 14.6s
7:	learn: 0.5297535	total: 108ms	remaining: 13.4s
8:	learn: 0.5133560	total: 112ms	remaining: 12.4s
9:	learn: 0.4950410	total: 118ms	remaining: 11.7s
10:	learn: 0.4790166	total: 123ms	remaining: 11s
11:	learn: 0.4641823	total: 127ms	remaining: 10.5s
12:	learn: 0.4482322	total: 133ms	remaining: 10.1s
13:	learn: 0.4335402	total: 137ms	remaining: 9.67s
14:	learn: 0.4186347	total: 142ms	remaining: 9.34s
15:	learn: 0.4077308	total: 149ms	remaining: 9.16s
16:	learn: 0.3971588	total: 155ms	remaining: 8.97s
17:	learn: 0.3882594	total: 160ms	remaining: 8.73s
18:	learn: 0.3776414

In [7]:
for key, value in performance_list.items():
    print(f"Regions: {key}, Acc: {value}")

Regions: 10, Acc: 0.9768250289687138
Regions: 20, Acc: 0.9895712630359212
Regions: 30, Acc: 0.9907300115874855
Regions: 40, Acc: 0.9930475086906141
Regions: 50, Acc: 0.9942062572421785
Regions: 60, Acc: 0.9918887601390498
Regions: 70, Acc: 0.9918887601390498
Regions: 80, Acc: 0.9907300115874855
Regions: 90, Acc: 0.9895712630359212
Regions: 100, Acc: 0.9953650057937428
Regions: 110, Acc: 0.9942062572421785
Regions: 120, Acc: 0.9930475086906141


Based on the observation, keeping the top 40 regions makes the most sense

In [ ]:
print("The top 40 features are: ")
print(len(feature_names[:40]))
print(feature_names[:40])

The top 40 features are: 
40
[0, 36, 82, 49, 12, 24, 14, 83, 117, 16, 45, 44, 35, 26, 47, 42, 72, 55, 30, 77, 67, 57, 88, 37, 23, 3, 108, 21, 85, 112, 69, 60, 93, 81, 38, 7, 62, 29, 61, 56]
